In [ ]:
import sys
sys.path.append("..")

import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from training import (
    prepare_dataset,
    evaluate_tool_calling_accuracy,
    extract_tool_calls_from_text,
)

In [ ]:

model_id = "../adapters/ep3_r8_a16_lr8e-04_bs8_best"
# Define tool schema to be parsed in the chat template
bt_tool = [{
            "name": "execute_behavior_tree",
            "arguments": {
                "bt_xml_filename": "<selected_bt>.xml",
                "execution_id": "<your numeric agent id>",
                "input_parameters": { "arg1": "value1" }
            }
        }]
model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto",
    )
processor = AutoProcessor.from_pretrained(model_id)


In [ ]:
# Load custom chat template from .jinja file
with open("../templates/qwen3.jinja", "r", encoding="utf-8") as f:
    custom_template = f.read()
processor.chat_template = custom_template

# Prepare dataset
train_dataset, eval_dataset, raw_eval_data = prepare_dataset(
    json_path="../data/train_dataset.json",  
    system_prompt_path="../data/system_prompt.txt",  
    processor=processor,
    train_split=0.8,
    tools=bt_tool
)

In [ ]:

# Custom evaluation for tool calling accuracy
print("\n Evaluating tool calling accuracy...")
eval_results = evaluate_tool_calling_accuracy(
    model, 
    raw_eval_data,
    processor,
    tools=bt_tool
)

In [ ]:
# Find where the assistant should respond
generated_chat = []
messages = raw_eval_data[0]  # Take the first example for demonstration
print(messages)
for i, msg in enumerate(messages):
    if msg["role"] == "assistant" and "tool_calls" in msg:
        # This is the point where we want the model to generate
        context_text = processor.apply_chat_template(
            generated_chat,
            tools = bt_tool,
            tokenize=False,
            add_generation_prompt=False,
            continue_final_message=False,
            enable_thinking=False
        )
        inputs = processor(context_text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1024,
                temperature=0.1,  # low for deterministic evaluation
                do_sample=False,
            )

        generated_text = processor.decode(outputs[0], skip_special_tokens=False)
        # Extract the model response part from the generated text (after the context)
        model_response = generated_text[len(context_text):].strip()
        # extract tool call
        tool_calls = extract_tool_calls_from_text(model_response)
        if tool_calls:
            generated_chat.append({
                "role": "assistant",
                "tool_calls": tool_calls
            })
        else:
            generated_chat.append({
                "role": "assistant",
                "content": model_response
            })
    else:
        generated_chat.append(msg)

final_template_text = processor.apply_chat_template(
            generated_chat,
            tools = bt_tool,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False
        )

print("\n" + "="*60)
print("MODEL GENERATED TEXT:")
print("="*60)
print(final_template_text)
print("="*60)
print("ORIGINAL DATA:")
print("="*60)
print(messages)